# Task 2.3.3 — Title embeddings with a pre-trained fastText model (Dask)

**Goal**: turn the title of every CORD-19 paper into a numerical vector (embedding),
using the pre-trained fastText model `crawl-300d-2M-subword.vec` (2M words, 300 dims).

**Input**: `silver/papers` (one row per paper, see `DATA_DICTIONARY.md`).
**Output**: one 300-dim vector per paper (`cord_uid`), saved as Parquet in
`/data/output_2.3.3/run_<N>workers/`.

## The key idea: the model is *data*, not an object

The model file is a 4.5 GB text file (`word v1 ... v300`, one word per line).
Loading it fully in RAM is impossible on our workers (3.8 GB each).
So we never load it: we treat it as a **distributed dataset**, like any other input.

Pipeline (each step is a Dask operation seen in the course):

```
silver/papers ──> tokenize titles ──> (cord_uid, word)      [dataframe, narrow ops]
                          │
                          └──> unique words = VOCABULARY    [small, to the driver]
                                        │
model.vec ──> read in ~64MB blocks ──> keep only rows       [filter, narrow]
              (dd.read_csv)            whose word is in
                                       the vocabulary
                                        │
(cord_uid, word) ⋈ (word, vector) ──> groupby(cord_uid)     [join + tree reduction]
                                       mean of vectors
                                        │
                                        └──> Parquet output + benchmark files
```

The average of the word vectors ("mean pooling") gives one **fixed-size vector per
title**, which is the aggregated representation explicitly allowed by the assignment
and the practical input for the cosine-similarity task (2.3.4).


## 1 · Configuration

Paths are the ones of the cluster (`/data` is the shared volume, mounted on every
node via NFS — run `scripts/cluster_storage_up.sh` first).
Every knob can be overridden with an environment variable, but the defaults are
already tuned for our VMs (2 vCPU / 3.8 GB each).

Cluster nodes are read from **`cluster.txt`** (same convention as the rest of the
repo): one line, `ip_scheduler,ip_worker1,ip_worker2,...` — the first host is the
scheduler, all the following ones become workers. If the file does not exist the
notebook falls back to a `LocalCluster` (useful for a quick local try).


In [ ]:
import os

# --- input data (shared volume, same absolute path on every node) ---
PAPERS_PATH = os.environ.get("EMB_PAPERS", "/data/MAPD-Project/data/silver/papers")
MODEL_PATH  = os.environ.get("EMB_MODEL",  "/data/model/crawl-300d-2M-subword.vec")

# --- output root (on the shared volume, NOT inside the git repo) ---
OUTPUT_ROOT = os.environ.get("EMB_OUTPUT", "/data/output_2.3.3")

# --- tuning knobs ---
BLOCKSIZE   = os.environ.get("EMB_BLOCKSIZE", "64MB")   # size of one model chunk read by a task
THREADS     = int(os.environ.get("EMB_THREADS", "2"))    # threads per worker (= vCPUs of our VMs)
MEMORY_LIM  = os.environ.get("EMB_MEMORY_LIMIT", "3GB")  # per worker, leaves ~0.8GB to the OS
SPLIT_OUT   = int(os.environ.get("EMB_SPLIT_OUT", "8"))  # output partitions of the final groupby
N_DIM       = 300                                        # fastText vector size


def read_cluster_hosts():
    """Read 'ip_scheduler,ip_worker1,...' from cluster.txt (cwd or parent dir).
    Returns None if the file is missing -> LocalCluster."""
    for folder in (os.getcwd(), os.path.dirname(os.getcwd())):
        path = os.path.join(folder, "cluster.txt")
        if os.path.exists(path):
            for line in open(path):
                line = line.strip()
                if line and not line.startswith("#"):
                    return [h.strip() for h in line.split(",") if h.strip()]
    return None


HOSTS = read_cluster_hosts()
print("papers :", PAPERS_PATH)
print("model  :", MODEL_PATH)
print("output :", OUTPUT_ROOT)
print("hosts  :", HOSTS if HOSTS else "(no cluster.txt -> LocalCluster)")


## 2 · Start the Dask cluster

Same `SSHCluster` convention used in the course and in `conversion_sanification.ipynb`.

Two details taken from our own experience on this cluster
(`docs/MEMORY_LEAK_REPORT.md`):

1. **glibc environment at worker spawn** (`MALLOC_TRIM_THRESHOLD_`, `MALLOC_ARENA_MAX`):
   makes the C allocator return freed memory to the OS instead of hoarding it.
   It must be set through `distributed.nanny.pre-spawn-environ` **before** the
   cluster is created — setting it later has no effect.
2. The cell is **idempotent**: re-running it closes the previous cluster first.


In [ ]:
import dask
from dask.distributed import Client, LocalCluster, wait

# (1) allocator env for the workers - must be set BEFORE creating the cluster
_env = dict(dask.config.get("distributed.nanny.pre-spawn-environ"))
_env.update({"MALLOC_TRIM_THRESHOLD_": 0, "MALLOC_ARENA_MAX": 2})
dask.config.set({"distributed.nanny.pre-spawn-environ": _env})

# (2) idempotency: close what is already running before creating a new cluster
try:
    client.close(); cluster.close()
except NameError:
    pass

if HOSTS:
    # multi-node cluster started via SSH (course method).
    # HOSTS[0] = scheduler only; every following host becomes a worker.
    from dask.distributed import SSHCluster
    cluster = SSHCluster(
        HOSTS,
        connect_options={"known_hosts": None},
        worker_options={"nthreads": THREADS, "memory_limit": MEMORY_LIM},
        scheduler_options={"port": 8786, "dashboard_address": ":8787"},
    )
    client = Client(cluster)
else:
    # local fallback (development on a laptop)
    cluster = LocalCluster(n_workers=2, threads_per_worker=THREADS,
                           memory_limit=MEMORY_LIM, processes=True)
    client = Client(cluster)

N_WORKERS = len(client.scheduler_info()["workers"])

# every run writes into a folder named after the number of workers,
# so benchmark runs with different cluster sizes never overwrite each other
RUN_DIR = os.path.join(OUTPUT_ROOT, f"run_{N_WORKERS}workers")
os.makedirs(RUN_DIR, exist_ok=True)

print("dashboard :", client.dashboard_link)
print("workers   :", N_WORKERS, "| run dir:", RUN_DIR)
client


## 3 · Imports and small helpers

- `stage()` / `timings`: a minimal stopwatch, one entry per pipeline stage
  (this is what the benchmark saves at the end).
- `sweep()`: asks every worker to return freed memory to the OS
  (`gc` + Arrow pool + `malloc_trim`). We call it **between** stages, never inside
  a timed region (recipe from `MEMORY_LEAK_REPORT.md` §7).
- `MemorySampler`: records the memory of the whole cluster over time, per stage.


In [ ]:
import csv as csv_mod
import json
import time

import numpy as np
import pandas as pd
import pyarrow as pa

import dask.dataframe as dd
from distributed.diagnostics import MemorySampler

timings = {}          # stage name -> seconds
ms = MemorySampler()  # cluster memory over time


def stage(name, t0):
    """Store and print the wall time of one pipeline stage."""
    dt = time.perf_counter() - t0
    timings[name] = round(dt, 2)
    print(f"[{name}] done in {dt:.1f}s")


def sweep():
    """Return freed memory to the OS on every worker (run between stages)."""
    def _sweep():
        import gc, ctypes, ctypes.util
        import pyarrow as _pa
        gc.collect()
        _pa.default_memory_pool().release_unused()
        try:
            ctypes.CDLL(ctypes.util.find_library("c")).malloc_trim(0)
        except Exception:
            pass
    client.run(_sweep)


## 4 · Stage A — from titles to (cord_uid, word) pairs

We read only the columns we need from `silver/papers` (Parquet is columnar, so
nothing else is even touched on disk) and keep the papers with a usable title
(`title_ok`, see `DATA_DICTIONARY.md`).

Tokenization is done on `title_norm` (already lower-case and whitespace-clean):

1. `str.findall('[a-z]{2,}')` extracts alphabetic words of length >= 2
   (vectorized pandas, no Python loop);
2. `explode` turns the list of words into one row per word
   (same idea as `flatMap` on a Bag);
3. a small list of English **stop-words** is removed: they carry no meaning and
   would dominate the average vector.

The result is small (~few 10s of MB), so we `persist()` it: it is used twice
(vocabulary now, join later) and we do not want to recompute it.


In [ ]:
# ~60 very common English words, excluded from the embeddings
STOPWORDS = [
    "the", "of", "and", "in", "to", "a", "an", "is", "are", "was", "were", "be",
    "for", "on", "with", "by", "as", "at", "from", "that", "this", "these",
    "those", "it", "its", "or", "not", "no", "but", "if", "then", "than", "so",
    "such", "can", "could", "may", "might", "will", "would", "shall", "should",
    "has", "have", "had", "do", "does", "did", "we", "our", "you", "your", "they",
    "their", "he", "she", "his", "her", "i", "my", "me", "us", "am", "been",
    "being", "into", "over", "under", "between", "during", "through", "about",
]

t0 = time.perf_counter()
with ms.sample("A_tokenize"):

    # read only the needed columns; keep papers with a usable title
    papers = dd.read_parquet(
        PAPERS_PATH,
        columns=["cord_uid", "title", "title_norm", "title_ok", "is_title_unique"],
    )
    papers = papers[papers["title_ok"]]

    # one row per (paper, word): findall -> list of words, explode -> rows
    tokens = papers[["cord_uid", "title_norm"]].copy()
    tokens["word"] = tokens["title_norm"].str.findall(r"[a-z]{2,}")
    tokens = tokens[["cord_uid", "word"]].explode("word")
    tokens = tokens[~tokens["word"].isin(STOPWORDS)]
    tokens = tokens.dropna()

    tokens = tokens.persist()   # small; reused twice (vocabulary + join)
    wait(tokens)

stage("A_tokenize", t0)         # stop the clock BEFORE the informative counts below
sweep()

n_titles = len(papers)
n_tokens = len(tokens)
print(f"titles with a usable title : {n_titles:,}")
print(f"(cord_uid, word) pairs     : {n_tokens:,}")
tokens.head()


## 5 · Stage B — read the model and keep only the useful words

`dd.read_csv` reads the 4.5 GB `.vec` file in ~64 MB blocks: **each worker parses
only its own blocks**, nobody ever holds the whole model. The file is
space-separated (`word v1 ... v300`), the first line is a header to skip.
Reading options that matter:

- `quoting=QUOTE_NONE`: some fastText "words" are punctuation (`"` , `,` ...) —
  they must not be treated as CSV quotes;
- `keep_default_na=False`: the literal word `nan`/`null` must stay a string;
- `dtype float32`: halves the memory of the 300 numeric columns.

**Filtering (the important trick).** We only need the ~10⁵ words that actually
appear in the titles, out of 2M. The naive `series.isin(python_set)` re-converts
the whole set on **every** partition — this is exactly the memory-churn hotspot
documented in `docs/MEMORY_LEAK_REPORT.md`. The cure ("fast-isin") is to build an
Arrow array from the vocabulary **once on the driver**, and use the vectorized
Arrow kernel `pc.is_in` inside each partition. The filter is a *narrow* operation:
each model block is reduced locally, no shuffle happens here.

Only the surviving slice (~200 MB) is persisted in the cluster memory.


In [ ]:
t0 = time.perf_counter()
with ms.sample("B_model_filter"):

    # ---- vocabulary: unique title words, brought to the driver (small) ----
    vocab = tokens["word"].unique().compute()
    vocab_arr = pa.array(sorted(vocab), type=pa.string())   # built ONCE (fast-isin)

    # ---- lazy definition of the model dataframe: 1 + 300 columns ----
    VCOLS = [f"v{i}" for i in range(N_DIM)]
    model = dd.read_csv(
        MODEL_PATH,
        sep=" ",
        header=None,
        skiprows=1,                       # first line = "2000000 300"
        names=["word"] + VCOLS,
        dtype={c: "float32" for c in VCOLS},
        quoting=csv_mod.QUOTE_NONE,       # do not interpret quote characters
        keep_default_na=False,            # keep the literal word "nan" as a string
        encoding="utf-8",
        blocksize=BLOCKSIZE,
    )

    def keep_vocab_words(pdf):
        """Keep only the rows whose word appears in the title vocabulary.
        Vectorized Arrow kernel; vocab_arr was converted once on the driver."""
        import pyarrow as _pa
        import pyarrow.compute as pc
        words = _pa.array(pdf["word"], from_pandas=True)
        mask = pc.is_in(words, value_set=vocab_arr).to_numpy(zero_copy_only=False)
        return pdf[mask]

    model_f = model.map_partitions(keep_vocab_words, meta=model._meta)
    model_f = model_f.persist()           # small slice of the model stays in RAM
    wait(model_f)

stage("B_model_filter", t0)     # stop the clock BEFORE the informative counts below
sweep()

n_vocab = len(vocab)
n_model_rows = len(model_f)
print(f"vocabulary size            : {n_vocab:,}")
print(f"model words kept           : {n_model_rows:,}  ({n_model_rows/n_vocab:.1%} of the vocabulary)")


## 6 · Stage C — join, average, write

1. **Join** `(cord_uid, word)` with `(word, vector)` — inner: words that are not
   in the model are simply skipped, as suggested by the assignment.
2. **Mean pooling as a map-reduce**: instead of collecting the list of vectors per
   title, we add a counter column `n = 1` and compute a single distributed
   `groupby(cord_uid).sum()` — Dask reduces partial sums inside each partition and
   then combines them (a tree reduction, same logic as `foldby` on Bags).
   The mean is then `sum / n`, computed column-wise. One shuffle in total.
3. We attach `title` and `is_title_unique` (useful for task 2.3.4) and write the
   result as Parquet (zstd) into the run folder.

The output schema is: `cord_uid | title | is_title_unique | n_words | v0..v299`.


In [ ]:
t0 = time.perf_counter()
with ms.sample("C_join_reduce_write"):

    # inner join: unmatched words are dropped (assignment hint)
    merged = tokens.merge(model_f, on="word", how="inner")

    # mean pooling via a single groupby-sum (tree reduction) + division
    merged["n"] = np.float32(1.0)
    sums = merged.groupby("cord_uid")[VCOLS + ["n"]].sum(split_out=SPLIT_OUT)

    emb = sums[VCOLS].div(sums["n"], axis=0).astype("float32")
    emb["n_words"] = sums["n"].astype("int32")
    emb = emb.reset_index()

    # attach title + duplicate-title flag for the next task (2.3.4)
    flags = papers[["cord_uid", "title", "is_title_unique"]]
    emb = emb.merge(flags, on="cord_uid", how="left")
    emb = emb[["cord_uid", "title", "is_title_unique", "n_words"] + VCOLS]

    out_path = os.path.join(RUN_DIR, "embeddings")
    emb.to_parquet(out_path, compression="zstd", write_index=False, overwrite=True)

stage("C_join_reduce_write", t0)
sweep()
print("embeddings written to:", out_path)


## 7 · Coverage summary and benchmark files

Statistics are computed **from the written output** (reading back only the tiny
`n_words` column), so nothing of the pipeline is recomputed. Everything is saved
in the run folder:

- `timings.csv` — wall time of each stage (input of the scaling comparison, §10);
- `summary.json` — run metadata (workers, sizes, coverage, ...);
- `memory_usage.png` — cluster memory over time, one curve per stage (§8).


In [ ]:
# read back only one small column to get the output statistics
check = dd.read_parquet(out_path, columns=["n_words"])
n_titles_out = len(check)
n_tokens_matched = int(check["n_words"].sum().compute())

summary = {
    "n_workers": N_WORKERS,
    "threads_per_worker": THREADS,
    "memory_limit": MEMORY_LIM,
    "model_blocksize": BLOCKSIZE,
    "groupby_split_out": SPLIT_OUT,
    "titles_in": int(n_titles),
    "titles_embedded": int(n_titles_out),
    "titles_coverage": round(n_titles_out / n_titles, 4),
    "token_pairs_in": int(n_tokens),
    "token_pairs_matched": int(n_tokens_matched),
    "token_coverage": round(n_tokens_matched / n_tokens, 4),
    "vocabulary_size": int(n_vocab),
    "model_words_kept": int(n_model_rows),
    "timings_seconds": timings,
    "total_seconds": round(sum(timings.values()), 2),
    "dask_version": dask.__version__,
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(os.path.join(RUN_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

with open(os.path.join(RUN_DIR, "timings.csv"), "w", newline="") as f:
    w = csv_mod.writer(f)
    w.writerow(["stage", "seconds", "n_workers"])
    for k, v in timings.items():
        w.writerow([k, v, N_WORKERS])

print(json.dumps(summary, indent=2))


## 8 · Memory of the cluster during the run

One curve per stage. This is how we verified on the real cluster that the memory
reaches a plateau (working set) instead of growing without bound — see
`docs/MEMORY_LEAK_REPORT.md` §7.4 for how to read this plot.


In [ ]:
import matplotlib.pyplot as plt

ms.plot(align=True, grid=True)
plt.title(f"Cluster memory per stage — {N_WORKERS} workers")
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "memory_usage.png"), dpi=120)
plt.show()


## 9 · Sanity check — do the embeddings make sense?

We read three titles back: two about coronaviruses and one unrelated, and compute
their cosine similarity on the driver with plain NumPy. Related titles must score
clearly higher than unrelated ones. (The full pairwise computation is task 2.3.4.)


In [ ]:
emb_out = dd.read_parquet(out_path)

# two related titles and one unrelated title (searched across all partitions)
covid = emb_out[emb_out["title"].str.contains(
    "coronavirus", case=False, na=False)].head(2, npartitions=-1)
other = emb_out[~emb_out["title"].str.contains(
    "coronavirus|covid|sars|virus|viral|infection",
    case=False, na=False)].head(1, npartitions=-1)

a = covid.iloc[0]
b = covid.iloc[1]
c = other.iloc[0]

def cosine(u, v):
    u, v = u[VCOLS].to_numpy(float), v[VCOLS].to_numpy(float)
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print("A:", a["title"][:90])
print("B:", b["title"][:90])
print("C:", c["title"][:90])
print()
print(f"cos(A, B)  related   = {cosine(a, b):.3f}")
print(f"cos(A, C)  unrelated = {cosine(a, c):.3f}")


## 10 · Scaling with the number of workers

Each run of this notebook writes its timings into `run_<N>workers/timings.csv`.
To measure scaling: edit `cluster.txt` (e.g. remove one worker IP), restart the
kernel and re-run the notebook — a new folder is produced automatically.
This cell collects every folder found so far and draws the comparison.


In [ ]:
import glob

rows = []
for path in sorted(glob.glob(os.path.join(OUTPUT_ROOT, "run_*workers", "timings.csv"))):
    rows.append(pd.read_csv(path))

if rows:
    bench = pd.concat(rows, ignore_index=True)
    pivot = bench.pivot_table(index="n_workers", columns="stage", values="seconds")
    pivot["total"] = pivot.sum(axis=1)
    print(pivot)

    pivot.plot(kind="bar", figsize=(8, 4))
    plt.ylabel("wall time (s)")
    plt.xlabel("number of workers")
    plt.title("Task 2.3.3 — wall time per stage vs number of workers")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_ROOT, "scaling_workers.png"), dpi=120)
    plt.show()
else:
    print("no timings.csv found yet")


## 11 · Shutdown

Closing the notebook does **not** stop the cluster: always run this cell at the end.


In [ ]:
client.close()
cluster.close()
print("cluster closed")
